# Step 6 — Train/Test Split & Data Preparation
### Credit Card Underwriting Pipeline

---

## Why We Split Data

> **The cardinal rule of machine learning:** Never evaluate a model on data it was trained on.
>
> If you train and test on the same data, the model has already "seen the answers."
> It memorises the training data instead of learning general patterns.
> This is called **overfitting** — great on training data, terrible on new data.

**The correct process:**
1. Split data into **training set** (model learns here) and **test set** (evaluation only)
2. The model never sees the test set during training
3. After training, evaluate on the test set to get an honest performance estimate

---

## Topics in This Notebook

| Section | Topic | Why |
|---------|-------|-----|
| 6.1 | Stratified Train/Test Split | Preserve class ratio in both sets |
| 6.2 | Why Stratification Matters | Avoid accidental imbalance in small datasets |
| 6.3 | SMOTE — Handle Class Imbalance | Make the model pay equal attention to both classes |
| 6.4 | StandardScaler | Normalise feature scales |
| 6.5 | Why Order Matters | Fit on train only, transform both |


In [ ]:
# ── Full pipeline re-run (Steps 1-5) ─────────────────────────────────────────
import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid')
SEED = 42

DATA_PATH   = 'cc_underwriting_5k_stratified11.csv'
IGNORE_COLS = ['applicant_id','target_approved','target_credit_limit_assigned']

df_raw       = pd.read_csv(DATA_PATH)
numeric_cols = [c for c in df_raw.select_dtypes(include='number').columns if c not in IGNORE_COLS]
cat_cols     = [c for c in df_raw.select_dtypes(include=['object','string']).columns if c not in IGNORE_COLS]

def missing_report(df):
    m=df.isnull().sum(); p=m/len(df)*100
    r=pd.DataFrame({'missing_count':m,'missing_pct':p.round(2)})
    return r[r['missing_count']>0].sort_values('missing_pct',ascending=False)

miss = missing_report(df_raw)
drop_cols = miss[miss['missing_pct']>50].index.tolist()
df = df_raw.drop(columns=drop_cols)
numeric_cols = [c for c in numeric_cols if c in df.columns]
cat_cols     = [c for c in cat_cols     if c in df.columns]
for col in numeric_cols:
    if df[col].isnull().sum()>0: df[col].fillna(df[col].median(),inplace=True)
for col in cat_cols:
    if df[col].isnull().sum()>0: df[col].fillna(df[col].mode()[0],inplace=True)
df['target'] = (df['target_approved']=='Yes').astype(int)

# Step 5 features
df_fe = df.copy(); eps=1e-6
df_fe['income_to_limit_ratio']   = df_fe['annual_income']/(df_fe['requested_credit_limit']+eps)
df_fe['expenses_to_income_ratio']= df_fe['total_monthly_expenses']/(df_fe['monthly_income']+eps)
df_fe['savings_to_income_ratio'] = df_fe['savings_account_balance']/(df_fe['annual_income']+eps)
df_fe['assets_to_liabilities']   = df_fe['total_assets']/(df_fe['total_liabilities']+eps)
df_fe['bureau_score_mean']       = df_fe[['fico_score','equifax_score','experian_score','transunion_score']].mean(axis=1)
df_fe['bureau_score_std']        = df_fe[['fico_score','equifax_score','experian_score','transunion_score']].std(axis=1)
df_fe['credit_history_yrs']      = df_fe['credit_history_length_months']/12
df_fe['utilization_x_inquiries'] = df_fe['credit_utilization_ratio']*df_fe['hard_inquiries_last_12mo']
df_fe['derogatory_x_dti']        = df_fe['derogatory_marks_count']*df_fe['debt_to_income_ratio']
df_fe['monthly_net_income']      = df_fe['monthly_income']-df_fe['total_monthly_expenses']
df_fe['income_per_dependent']    = df_fe['annual_income']/(df_fe['dependents_count']+1)
df_fe['income_yrs_employed']     = df_fe['annual_income']*df_fe['years_employed']
risk_flags=['prior_default_flag','prior_bankruptcy_flag','high_risk_industry_flag',
            'recent_address_change_flag','recent_employment_change_flag',
            'multiple_applications_flag','thin_file_flag','no_hit_flag']
for f in risk_flags:
    if f in df_fe.columns: df_fe[f+'_enc']=(df_fe[f]=='Yes').astype(int)
df_fe['total_risk_flags']=df_fe[[f+'_enc' for f in risk_flags if f+'_enc' in df_fe.columns]].sum(axis=1)
df_fe['age_squared']=df_fe['age']**2; df_fe['age_x_fico']=df_fe['age']*df_fe['fico_score']
SKEWED=['annual_income','monthly_income','net_worth','total_assets',
        'savings_account_balance','requested_credit_limit','total_liabilities']
for col in SKEWED:
    if col in df_fe.columns: df_fe[col+'_log']=np.log1p(np.clip(df_fe[col],0,None))
le=LabelEncoder()
cat_for_model=[c for c in cat_cols if df_fe[c].nunique()<=30]
for col in cat_for_model: df_fe[col+'_enc']=le.fit_transform(df_fe[col].astype(str))
NEW_FEATURES=['income_to_limit_ratio','expenses_to_income_ratio','savings_to_income_ratio',
              'assets_to_liabilities','bureau_score_mean','bureau_score_std','credit_history_yrs',
              'utilization_x_inquiries','derogatory_x_dti','monthly_net_income',
              'income_per_dependent','income_yrs_employed','total_risk_flags','age_squared','age_x_fico']
LOG_FEATURES=[c+'_log' for c in SKEWED if c+'_log' in df_fe.columns]
CAT_ENC=[c+'_enc' for c in cat_for_model]
ALL_FEATURES=list(dict.fromkeys(numeric_cols+NEW_FEATURES+LOG_FEATURES+CAT_ENC))
ALL_FEATURES=[f for f in ALL_FEATURES if f in df_fe.columns]
FINAL_FEATURES=ALL_FEATURES  # simplified for this notebook
X = df_fe[FINAL_FEATURES].copy().fillna(0)
y = df_fe['target'].copy()
print(f'Pipeline complete. X: {X.shape}, y: {y.value_counts().to_dict()}')

## 6.1 — Stratified Train/Test Split

### What is Stratification?

> **Without stratification:** The random split might put 80% of all "Declined" cases in training.
> Then the test set has very few declines, and our evaluation metrics are unreliable.
>
> **With stratification (`stratify=y`):** Both train and test sets have the same class ratio
> as the original data. If the full dataset is 65% Approved, both train and test will be ~65% Approved.

### Why 80/20?

- **80% training** → enough data for the model to learn patterns
- **20% testing** → enough data for reliable evaluation metrics
- Industry standard. For very small datasets, consider 90/10 or K-fold cross-validation.


In [ ]:
# Stratified 80/20 split
# stratify=y ensures the class ratio is preserved in both splits
# random_state=SEED ensures reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,       # 20% held out for evaluation
    random_state=SEED,    # reproducible split
    stratify=y            # preserve class ratio
)

print('Train/Test Split Results:')
print(f'  Training rows : {X_train.shape[0]:,}  ({X_train.shape[0]/len(X):.0%} of data)')
print(f'  Test rows     : {X_test.shape[0]:,}   ({X_test.shape[0]/len(X):.0%} of data)')
print()
print('Class distribution in each split:')
print(f'  Original  — Approved: {y.mean():.1%}  Declined: {(1-y.mean()):.1%}')
print(f'  Training  — Approved: {y_train.mean():.1%}  Declined: {(1-y_train.mean()):.1%}')
print(f'  Test      — Approved: {y_test.mean():.1%}  Declined: {(1-y_test.mean()):.1%}')
print()
print('The class ratios are preserved in all splits — stratification worked!')

## 6.2 — SMOTE: Handling Class Imbalance

### The Problem

> Our dataset has ~65% Approved, ~35% Declined.
> A naive model could achieve 65% accuracy by **always predicting Approved**.
> It would never catch a single bad applicant — catastrophic for a bank!

### SMOTE — Synthetic Minority Oversampling Technique

**How SMOTE works:**
1. For each minority class (Declined) sample, find its K nearest neighbors (also Declined)
2. Randomly pick one neighbor
3. Create a new synthetic sample somewhere along the line between the sample and that neighbor
4. Repeat until the classes are balanced

> **Critical rule: Apply SMOTE ONLY to the training set, NEVER to the test set.**
> The test set must represent the real-world distribution (imbalanced).
> If you SMOTE the test set, your evaluation metrics are invalid.

**Alternatives to SMOTE:**
- `class_weight='balanced'` in scikit-learn (tells the model to weight minority class more)
- Random undersampling (drop majority class rows — data loss)
- ADASYN (adaptive SMOTE — focuses on harder-to-classify minority samples)


In [ ]:
# Visualise class imbalance before SMOTE
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Before SMOTE
before_counts = y_train.value_counts()
axes[0].bar(['Declined (0)', 'Approved (1)'],
            [before_counts.get(0,0), before_counts.get(1,0)],
            color=['#e74c3c', '#2ecc71'], edgecolor='white')
axes[0].set_title('Training Set\nBefore SMOTE', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate([before_counts.get(0,0), before_counts.get(1,0)]):
    axes[0].text(i, v+10, str(v), ha='center', fontweight='bold')

# Apply SMOTE
print('Applying SMOTE to training data...')
print(f'Before — Approved: {y_train.sum()}, Declined: {(y_train==0).sum()}')

smote = SMOTE(
    random_state=SEED,
    k_neighbors=5    # number of nearest neighbors to use when creating synthetic samples
)

# fit_resample: learn the minority class neighborhood AND generate synthetic samples
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

after_counts = pd.Series(y_train_sm).value_counts()
print(f'After  — Approved: {after_counts.get(1,0)}, Declined: {after_counts.get(0,0)}')

# After SMOTE
axes[1].bar(['Declined (0)', 'Approved (1)'],
            [after_counts.get(0,0), after_counts.get(1,0)],
            color=['#e74c3c', '#2ecc71'], edgecolor='white')
axes[1].set_title('Training Set\nAfter SMOTE', fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate([after_counts.get(0,0), after_counts.get(1,0)]):
    axes[1].text(i, v+10, str(v), ha='center', fontweight='bold')

# Test set stays untouched
test_counts = y_test.value_counts()
axes[2].bar(['Declined (0)', 'Approved (1)'],
            [test_counts.get(0,0), test_counts.get(1,0)],
            color=['#e74c3c', '#2ecc71'], edgecolor='white')
axes[2].set_title('Test Set\n(NO SMOTE — real distribution)', fontweight='bold')
axes[2].set_ylabel('Count')

plt.suptitle('SMOTE Effect on Class Balance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Training rows after SMOTE: {X_train_sm.shape[0]:,} (added {X_train_sm.shape[0]-X_train.shape[0]:,} synthetic rows)')

## 6.3 — StandardScaler: Feature Normalisation

### Why Scale Features?

> **Different features live on completely different scales:**
> - `fico_score`: 300–850
> - `annual_income`: 12,000–5,000,000
> - `credit_utilization_ratio`: 0.0–1.0
>
> Without scaling, a model may implicitly assign more importance to high-magnitude features
> simply because their numbers are larger — not because they are more predictive.

### StandardScaler Formula

$$x_{scaled} = \frac{x - \mu}{\sigma}$$

Where:
- $\mu$ = mean of the column (in training data)
- $\sigma$ = standard deviation of the column (in training data)

After scaling: each column has **mean = 0** and **standard deviation = 1**.

### The Golden Rule: Fit on Train, Transform Both

> **Always fit the scaler on TRAINING data only.**
>
> If you fit on the full dataset (including test), you are using test statistics during training.
> This is called **data leakage** — the model has seen the test set's distribution,
> which makes evaluation metrics unrealistically optimistic.
>
> **Correct flow:**
> 1. `scaler.fit(X_train)` — learn mean and std from training data only
> 2. `scaler.transform(X_train)` — scale training data
> 3. `scaler.transform(X_test)` — scale test data using TRAINING mean/std (NOT test's own stats)


In [ ]:
# StandardScaler: normalise features to zero-mean, unit-variance
scaler = StandardScaler()

# fit_transform on train: LEARN mean/std from training data AND apply scaling
X_train_sc = scaler.fit_transform(X_train_sm)

# transform on test: apply SAME mean/std learned from training (no re-fitting!)
X_test_sc  = scaler.transform(X_test)

# Keep as DataFrame for SHAP plots in Step 9
X_train_sc_df = pd.DataFrame(X_train_sc, columns=FINAL_FEATURES)
X_test_sc_df  = pd.DataFrame(X_test_sc,  columns=FINAL_FEATURES)

print('Scaling complete!')
print()

# Verify: after scaling, each column should have mean ~0 and std ~1 (in training data)
sample_cols = FINAL_FEATURES[:5]
print('Verification (first 5 features after scaling):')
print(f'  {"Feature":<40} {"Mean":>10} {"Std":>10}')
print('  ' + '-'*62)
for col in sample_cols:
    idx = FINAL_FEATURES.index(col)
    mean_val = X_train_sc[:, idx].mean()
    std_val  = X_train_sc[:, idx].std()
    print(f'  {col:<40} {mean_val:>10.6f} {std_val:>10.6f}')

print()
print('Mean values are ~0 and std values are ~1 as expected.')
print()
print('Final data shapes for modelling:')
print(f'  X_train_sc : {X_train_sc.shape}  (after SMOTE + scaling)')
print(f'  X_test_sc  : {X_test_sc.shape}   (original distribution, same scaling)')
print(f'  y_train_sm : {y_train_sm.shape}  (balanced after SMOTE)')
print(f'  y_test     : {y_test.shape}      (original imbalanced distribution)')

## Step 6 Complete — Data Preparation Summary

**What we did:**

| Step | Action | Key Decision |
|------|--------|-------------|
| Stratified split | 80% train, 20% test | stratify=y to preserve class ratio |
| SMOTE | Balance training data | Only on TRAIN — test stays real |
| StandardScaler | Normalise feature scales | Fit on train, transform both |

**Data shapes going into modelling:**
- `X_train_sc`: Scaled, SMOTE-balanced training features
- `X_test_sc`: Scaled test features (original class distribution)
- `y_train_sm`: Balanced training labels (50/50 after SMOTE)
- `y_test`: Original imbalanced test labels (used for honest evaluation)

**Next:** `07_Model_Selection.ipynb` — Why Random Forest? Baseline training + hyperparameter tuning
